# 1.5 Tiling、形状变换、切片与转置

前面几节已经完成逐元素、矩阵乘法和规约计算。本节把视角从“算什么”扩展到“数据如何组织”。真实算子开发中，除了数学计算，还要频繁处理 Tiling、reshape、切片和转置。

本节目标如下：

- 从直观角度理解 Tiling 分块。
- 理解 Tile Shape 不改变数学结果，只影响执行组织。
- 学习 reshape、切片、transpose 的常见使用场景。
- 通过 Q/K 拆分示例理解形状操作。
- 练习将输入切成 Q、K、V 三段。


---
## 前置说明

本节在第一个代码单元格中集中放置环境准备代码（import、设备选择、`RUN_MODE` 等）。后续每个可运行模块在定义 kernel 前只调用 `reset_pypto_notebook_state()` 重置编译状态，不再重复完整的环境初始化。

环境准备的具体含义参见 `01.01_chapter_intro.ipynb` §1。


## 直观理解：Tiling 就是把大 Tensor 切成小 Tile

如果一个二维 Tensor 非常大，一次性处理可能不方便。Tiling 可以先理解为“切块”：把大 Tensor 切成很多小 Tile，每次处理一小块。

在工程上，可以把它理解为一种分批处理策略：完整计算目标不变，但执行时会把数据拆成更适合硬件处理的小块。

因此，Tiling 不改变数学结果，只改变 NPU 底层怎么分块搬运和计算数据。


## 2. Tiling 的直观理解

Tiling 可以理解为把一个大 Tensor 拆成多个小 Tile。这样做的目的不是改变数学结果，而是让数据搬运和计算更适合 NPU 的片上存储和计算单元。

可以从三个层面理解 Tiling：

| 层面 | 含义 |
| --- | --- |
| 数学层面 | 不改变最终计算结果 |
| 数据层面 | 改变数据被分块读取、计算、写回的方式 |
| 性能层面 | 影响访存效率、缓存使用和计算单元利用率 |

因此，学习 PyPTO 时要区分两个问题：

- Tensor 表达描述“要计算什么”。
- Tile Shape 描述“如何组织执行”。


## 3. 向量 Tile 与 Cube Tile

在 PyPTO 初级阶段，最常见的 Tile 设置有两类：

```python
pypto.set_vec_tile_shapes(8, 8)
pypto.set_cube_tile_shapes([32, 32], [64, 64], [64, 64])
```

它们对应不同计算类型：

| API | 典型场景 |
| --- | --- |
| `set_vec_tile_shapes` | 逐元素、规约、激活函数等向量类计算 |
| `set_cube_tile_shapes` | 矩阵乘法等 Cube 类计算 |

如果一个算子同时包含矩阵乘法和 Softmax，通常会同时涉及 Cube 计算和 Vector 计算。后续 Attention 示例就会出现这种组合。


## 3.1 常见 Tiling 与 Transform 用法

Tiling 关注计算如何分块执行，Transform 关注 Tensor 如何被读取、重排、组合或写回。常见用法包括：`set_vec_tile_shapes` 设置向量类 Tile，`set_cube_tile_shapes` 设置矩阵乘法 Tile，`view` 创建局部观察窗口，`assemble` 把局部结果写回大 Tensor，`gather/scatter` 处理索引读写，`concat` 处理拼接，`transpose` 交换维度，`cast` 改变 dtype。

这些操作共同服务于一个目标：在真正计算前后，把 Tensor 组织成适合计算和输出的形态。Tile 改变的是执行组织，不应该改变数学结果；shape、slice、transpose 改变的是数据被观察和摆放的方式。


## 3.2 两类 Tile 的常见模式

Tiling 模块可以按下面几类模式理解。它们属于同一组问题：Tile Shape 只改变执行组织，不改变数学结果。

| 模式 | 代表场景 | 阅读重点 |
| --- | --- | --- |
| Cube Tile 基础配置 | 单次矩阵乘法 | `set_cube_tile_shapes` 要出现在 `matmul` 前 |
| Cube Tile 形状变化 | 不同输入 shape、batched matmul | 输出 shape 由矩阵乘法规则决定，tile shape 不改变结果 |
| Cube Tile 性能观察 | 不同 cube tile 的 runtime 对比 | 耗时可变，正确性不应变 |
| Vec Tile 基础配置 | 逐元素加法、激活等向量类计算 | `set_vec_tile_shapes` 的维度数要和向量计算形状匹配 |
| Vec Tile 高维输入 | 3D / 4D Tensor | 每一维 tile 描述分块组织，不是原 Tensor 必须等大的 shape |
| Vec Tile 性能观察 | 不同 vec tile 的 runtime 对比 | runtime 是观察指标，数值一致性仍是第一检查项 |

闭环验证选取两类 Tile 的代表计算：逐元素加法和矩阵乘法。验证内容包括基础配置、不同输入 shape 和不同 tile shape 结果一致性；runtime 对比属于性能观察点，实际调优时再单独展开。


In [ ]:
import os
import time

# 环境准备：参见 01.01_chapter_intro.ipynb §1 的详细说明
# ASCEND_RT_VISIBLE_DEVICES 必须在 import torch 之前设置才生效。
# 如需指定其他 NPU，请在启动 Notebook 前设置 ASCEND_RT_VISIBLE_DEVICES。
# os.environ.setdefault("ASCEND_RT_VISIBLE_DEVICES", "13")

import torch
import pypto
from numpy.testing import assert_allclose

try:
    import torch_npu  # noqa: F401
except Exception:
    torch_npu = None


def reset_pypto_notebook_state():
    "Clean stale PyPTO recording state before defining a JIT kernel."
    try:
        pypto.reset()
    except Exception:
        pass

    try:
        from pypto._controller import Controller
        Controller.end_function()
    except Exception:
        pass


def get_device():
    if torch_npu is None:
        return "cpu"

    device_id = int(os.environ.get("TILE_FWK_DEVICE_ID", "0"))
    try:
        torch.npu.set_device(device_id)
    except Exception as exc:
        print("NPU device is not ready:", exc)
        return "cpu"
    return f"npu:{device_id}"


def max_abs_diff(actual, expected):
    return (actual.to(torch.float32) - expected.to(torch.float32)).abs().max().item()


reset_pypto_notebook_state()
device = get_device()
RUN_MODE = pypto.RunMode.NPU if device != "cpu" else pypto.RunMode.SIM

# print("ASCEND_RT_VISIBLE_DEVICES:", os.environ.get("ASCEND_RT_VISIBLE_DEVICES", "<未设置>"))
print("TILE_FWK_DEVICE_ID:", os.environ.get("TILE_FWK_DEVICE_ID", "<未设置，默认 0>"))
print("device:", device)

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def vec_add_tile_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32),
    tile_shapes: tuple):
    pypto.set_vec_tile_shapes(*tile_shapes)
    out.move(pypto.add(a, b))


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def vec_add_three_tile_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
    out1: pypto.Tensor([], pypto.DT_FP32),
    out2: pypto.Tensor([], pypto.DT_FP32),
    out3: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(1, 2, 8)
    out1.move(pypto.add(a, b))

    pypto.set_vec_tile_shapes(2, 6, 32)
    out2.move(pypto.add(a, b))

    pypto.set_vec_tile_shapes(5, 3, 16)
    out3.move(pypto.add(a, b))


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def cube_matmul_tile_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_cube_tile_shapes([32, 32], [64, 64], [64, 64])
    out.move(pypto.matmul(a, b, a.dtype))


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def cube_matmul_three_tile_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
    out1: pypto.Tensor([], pypto.DT_FP32),
    out2: pypto.Tensor([], pypto.DT_FP32),
    out3: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_cube_tile_shapes([32, 32], [16, 16], [32, 32])
    out1.move(pypto.matmul(a, b, a.dtype))

    pypto.set_cube_tile_shapes([32, 32], [16, 64], [32, 128])
    out2.move(pypto.matmul(a, b, a.dtype))

    pypto.set_cube_tile_shapes([64, 64], [128, 128], [128, 128])
    out3.move(pypto.matmul(a, b, a.dtype))


def main_tile_shape_coverage():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    # 1. Vec tile：3D 输入。
    v3a = torch.tensor([[[1, 2, 3], [1, 2, 3]]], dtype=torch.float32, device=device)
    v3b = torch.tensor([[[4, 5, 6], [4, 5, 6]]], dtype=torch.float32, device=device)
    v3out = torch.empty_like(v3a)
    vec_add_tile_kernel(v3a, v3b, v3out, (1, 2, 8))
    v3ref = v3a + v3b
    torch.testing.assert_close(v3out, v3ref, rtol=1e-3, atol=1e-3)

    # 2. Vec tile：4D 输入。
    v4a = torch.tensor([[[[1, 2, 3], [4, 5, 6]]]], dtype=torch.float32, device=device)
    v4b = torch.tensor([[[[7, 8, 9], [10, 11, 12]]]], dtype=torch.float32, device=device)
    v4out = torch.empty_like(v4a)
    vec_add_tile_kernel(v4a, v4b, v4out, (1, 1, 4, 8))
    v4ref = v4a + v4b
    torch.testing.assert_close(v4out, v4ref, rtol=1e-3, atol=1e-3)

    # 3. Vec tile：同一计算使用不同 tile shape，结果保持一致。
    vo1 = torch.empty_like(v3a)
    vo2 = torch.empty_like(v3a)
    vo3 = torch.empty_like(v3a)
    vec_add_three_tile_kernel(v3a, v3b, vo1, vo2, vo3)
    torch.testing.assert_close(vo1, v3ref, rtol=1e-3, atol=1e-3)
    torch.testing.assert_close(vo2, v3ref, rtol=1e-3, atol=1e-3)
    torch.testing.assert_close(vo3, v3ref, rtol=1e-3, atol=1e-3)

    # 4. Cube tile：基础矩阵乘法。
    m1 = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32, device=device)
    m2 = torch.tensor([[5, 6], [7, 8]], dtype=torch.float32, device=device)
    mm_out = torch.empty((2, 2), dtype=torch.float32, device=device)
    cube_matmul_tile_kernel(m1, m2, mm_out)
    mm_ref = torch.matmul(m1, m2)
    torch.testing.assert_close(mm_out, mm_ref, rtol=1e-3, atol=1e-3)

    # 5. Cube tile：batched matmul 使用不同 tile shape，结果保持一致。
    ba = torch.tensor([[[1, 2], [3, 4]], [[5, 6], [7, 8]]], dtype=torch.float32, device=device)
    bb = torch.tensor([[[5, 6], [7, 8]], [[1, 2], [3, 4]]], dtype=torch.float32, device=device)
    bo1 = torch.empty((2, 2, 2), dtype=torch.float32, device=device)
    bo2 = torch.empty((2, 2, 2), dtype=torch.float32, device=device)
    bo3 = torch.empty((2, 2, 2), dtype=torch.float32, device=device)
    cube_matmul_three_tile_kernel(ba, bb, bo1, bo2, bo3)
    bref = torch.matmul(ba, bb)
    torch.testing.assert_close(bo1, bref, rtol=1e-3, atol=1e-3)
    torch.testing.assert_close(bo2, bref, rtol=1e-3, atol=1e-3)
    torch.testing.assert_close(bo3, bref, rtol=1e-3, atol=1e-3)

    print("tiling 模块代表性验证通过")
    print("vec 3D shape:", tuple(v3out.shape), "最大误差:", max_abs_diff(v3out, v3ref), "样例:", v3out.flatten()[:4].cpu())
    print("vec 4D shape:", tuple(v4out.shape), "最大误差:", max_abs_diff(v4out, v4ref), "样例:", v4out.flatten()[:4].cpu())
    print("cube shape:", tuple(mm_out.shape), "最大误差:", max_abs_diff(mm_out, mm_ref), "输出:", mm_out.cpu())
    print("batched cube shape:", tuple(bo1.shape), "最大误差:", max_abs_diff(bo1, bref), "样例:", bo1[0].cpu())


main_tile_shape_coverage()


ASCEND_RT_VISIBLE_DEVICES: 13
TILE_FWK_DEVICE_ID: <未设置，默认 0>
device: npu:0
tiling 模块代表性验证通过
vec 3D shape: (1, 2, 3) 最大误差: 0.0 样例: tensor([5., 7., 9., 5.])
vec 4D shape: (1, 1, 2, 3) 最大误差: 0.0 样例: tensor([ 8., 10., 12., 14.])
cube shape: (2, 2) 最大误差: 0.0 输出: tensor([[19., 22.],
        [43., 50.]])
batched cube shape: (2, 2, 2) 最大误差: 0.0 样例: tensor([[19., 22.],
        [43., 50.]])


## 4. reshape：改变 Tensor 的观察方式

`reshape` 用于改变 Tensor 的形状，但要求元素总数保持一致。例如将 `[B, S, H]` 改为 `[B * S, H]`，常用于把 batch 和 sequence 两个维度合并后送入线性层。

示意代码：

```python
y = pypto.reshape(x, [batch * seq, hidden])
```

判断 reshape 是否合理时，先检查变换前后的元素数量是否一致。


### reshape 的直观理解

reshape 可以理解为“同一批数字，换一种摆放方式看”。

例如 6 个数字：

```python
[1, 2, 3, 4, 5, 6]
```

可以摆成 2 行 3 列：

```python
[[1, 2, 3],
 [4, 5, 6]]
```

也可以摆成 3 行 2 列：

```python
[[1, 2],
 [3, 4],
 [5, 6]]
```

数字总数仍然是 6。reshape 最重要的规则就是：变换前后元素总数要一致。


## 5. 切片：取出 Tensor 的局部区域

切片用于从 Tensor 中取出一段区域。例如输入 `x` 的最后一维包含 Q 和 K 两段数据，可以写成：

```python
q = x[:, :, :64]
k = x[:, :, 64:128]
```

这种写法常见于大模型中的 QKV 拆分。它不是计算密集操作，但会影响后续矩阵乘法和注意力计算如何组织输入。


### view、gather、scatter 的区别

`view`、`gather`、`scatter` 都和数据位置有关，但它们的定位不同。

`view` 适合取连续规则区域：

```python
pypto.view(input0, [tile_b, n, s, d], [b_offset, 0, 0, 0])
```

可以理解成：

```text
从 [b_offset, 0, 0, 0] 开始，取 shape 为 [tile_b, n, s, d] 的一块。
```

这里第二个参数是“取多大”，第三个参数是“从哪里取”。如果 `tile_b=2`，那么：

```python
pypto.view(input0, [2, 32, 1, 256], [4, 0, 0, 0])
```

对应直觉是：

```python
input0[4:6, :, :, :]
```

不能把第二个参数写成 `[b_offset, 32, 1, 256]`，因为 `b_offset` 是起始位置，不是局部块大小。

`gather` 适合按索引读取，不要求位置连续：

```text
input = [10, 20, 30, 40]
index = [2, 0, 3]
gather(input, index) -> [30, 10, 40]
```

`scatter` 适合按索引写入。典型调用形式如下：

```python
out[:] = pypto.scatter(x, dim, y, src)
```

这里的 `y` 就是索引 Tensor，也就是写入位置。更直观的名字可以写成：

```python
out[:] = pypto.scatter(x, dim, index_tensor, src)
```

三者可以这样区分：

| 操作 | 作用 |
| --- | --- |
| `view` | 按连续区域取块 |
| `gather` | 按 index 读取 |
| `scatter` | 按 index 写入 |


### 切片的直观理解

切片就像从一张大 Tensor里剪出一部分。例如 `x[:, :, :64]` 可以先读成：

“前面的维度都保留，最后一维只取前 64 个位置。”

其中冒号 `:` 表示“这一维全部要”，`:64` 表示“从开头取到第 64 个之前”。


## 5.1 view：用 shape 和 offsets 取局部窗口

`pypto.view(x, shape, offsets)` 可以理解成在原 Tensor 上开一个局部观察窗口。

- `shape` 表示窗口大小。
- `offsets` 表示窗口起点。

例如输入 shape 是 `[4, 8]`，写成 `pypto.view(x, [4, 4], [0, 4])`，表示从第 0 行、第 4 列开始，取一个 `[4, 4]` 的局部窗口。它等价于 PyTorch 里的 `x[0:4, 4:8]`。


In [26]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def view_window_kernel(
    x: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32)):
    tile_shapes = [8 for _ in range(len(x.shape))]
    pypto.set_vec_tile_shapes(*tile_shapes)
    out[:] = pypto.view(x, [4, 4], [0, 4])


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def view_with_valid_shape_kernel(
    x: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32)):
    tile_shapes = [8 for _ in range(len(x.shape))]
    pypto.set_vec_tile_shapes(*tile_shapes)
    out[:] = pypto.view(x, [4, 4], [2, 4], valid_shape=[2, 4])


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def assemble_offset_kernel(
    x: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32),
    offsets: list):
    tile_shapes = [8 for _ in range(len(x.shape))]
    pypto.set_vec_tile_shapes(*tile_shapes)
    pypto.assemble(x, offsets, out)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def transpose_kernel(
    x: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32)):
    tile_shapes = [8 for _ in range(len(x.shape))]
    pypto.set_vec_tile_shapes(*tile_shapes)
    out[:] = pypto.transpose(x, 0, 1)


LOOP_ASSEMBLE_SHAPE = (32, 32, 1, 256)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def add_loop_view_assemble_kernel(
    input0: pypto.Tensor(LOOP_ASSEMBLE_SHAPE, pypto.DT_FP32),
    input1: pypto.Tensor(LOOP_ASSEMBLE_SHAPE, pypto.DT_FP32),
    output: pypto.Tensor(LOOP_ASSEMBLE_SHAPE, pypto.DT_FP32)):
    pypto.set_vec_tile_shapes(1, 4, 1, 64)

    b, n, s, d = LOOP_ASSEMBLE_SHAPE
    tile_b = 1
    b_loop = b // tile_b

    for idx in pypto.loop(b_loop):
        b_offset = idx * tile_b
        left = pypto.view(input0, [tile_b, n, s, d], [b_offset, 0, 0, 0])
        right = pypto.view(input1, [tile_b, n, s, d], [b_offset, 0, 0, 0])
        partial = left + right
        pypto.assemble(partial, [b_offset, 0, 0, 0], output)


def main_transform_content_coverage():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    # 1. view：从第 0 行、第 4 列开始取 4x4 窗口。
    x = torch.tensor([
        [1, 1, 2, 2, 3, 3, 4, 4],
        [1, 1, 2, 2, 3, 3, 4, 4],
        [1, 1, 2, 2, 5, 5, 6, 6],
        [1, 1, 2, 2, 5, 5, 6, 6],
    ], dtype=torch.float32, device=device)
    view_out = torch.zeros((4, 4), dtype=torch.float32, device=device)
    view_window_kernel(x, view_out)
    view_ref = x[0:4, 4:8]
    torch.testing.assert_close(view_out, view_ref, rtol=1e-3, atol=1e-3)

    # 2. valid_shape：边界块只保留前 2 行有效数据。
    valid_out = torch.zeros((4, 4), dtype=torch.float32, device=device)
    view_with_valid_shape_kernel(x, valid_out)
    valid_ref = torch.tensor([
        [5, 5, 6, 6],
        [5, 5, 6, 6],
        [0, 0, 0, 0],
        [0, 0, 0, 0],
    ], dtype=torch.float32, device=device)
    torch.testing.assert_close(valid_out, valid_ref, rtol=1e-3, atol=1e-3)

    # 3. assemble：把 2x2 小块写到大 Tensor 的 [1, 1] 偏移处。
    block = torch.tensor([[2, 2], [2, 2]], dtype=torch.float32, device=device)
    assemble_out = torch.zeros((4, 4), dtype=torch.float32, device=device)
    assemble_offset_kernel(block, assemble_out, [1, 1])
    assemble_ref = torch.tensor([
        [0, 0, 0, 0],
        [0, 2, 2, 0],
        [0, 2, 2, 0],
        [0, 0, 0, 0],
    ], dtype=torch.float32, device=device)
    torch.testing.assert_close(assemble_out, assemble_ref, rtol=1e-3, atol=1e-3)

    # 4. transpose：二维转置，输出 shape 从 [2, 3] 变为 [3, 2]。
    trans_x = torch.tensor([[1.0028, -0.9893, 0.5809], [-0.1669, 0.7299, 0.4942]], dtype=torch.float32, device=device)
    trans_out = torch.empty((3, 2), dtype=torch.float32, device=device)
    transpose_kernel(trans_x, trans_out)
    trans_ref = trans_x.transpose(0, 1)
    torch.testing.assert_close(trans_out, trans_ref, rtol=1e-3, atol=1e-3)

    # 5. loop + view + assemble：按 batch 分块读、分块加、写回完整输出。
    loop_a = torch.rand(LOOP_ASSEMBLE_SHAPE, dtype=torch.float32, device=device)
    loop_b = torch.rand(LOOP_ASSEMBLE_SHAPE, dtype=torch.float32, device=device)
    loop_out = torch.empty(LOOP_ASSEMBLE_SHAPE, dtype=torch.float32, device=device)
    add_loop_view_assemble_kernel(loop_a, loop_b, loop_out)
    loop_ref = loop_a + loop_b
    torch.testing.assert_close(loop_out, loop_ref, rtol=3e-3, atol=3e-3)

    print("transform 模块代表性验证通过")
    print("view shape:", tuple(view_out.shape), "最大误差:", max_abs_diff(view_out, view_ref), "样例:", view_out[0].cpu())
    print("valid_shape shape:", tuple(valid_out.shape), "最大误差:", max_abs_diff(valid_out, valid_ref), "样例:", valid_out[:, 0].cpu())
    print("assemble shape:", tuple(assemble_out.shape), "最大误差:", max_abs_diff(assemble_out, assemble_ref), "输出:", assemble_out.cpu())
    print("transpose shape:", tuple(trans_out.shape), "最大误差:", max_abs_diff(trans_out, trans_ref), "输出:", trans_out.cpu())
    print("loop assemble shape:", tuple(loop_out.shape), "最大误差:", max_abs_diff(loop_out, loop_ref), "样例:", loop_out[0, 0, 0, :4].cpu())


main_transform_content_coverage()


transform 模块代表性验证通过
view shape: (4, 4) 最大误差: 0.0 样例: tensor([3., 3., 4., 4.])
valid_shape shape: (4, 4) 最大误差: 0.0 样例: tensor([5., 5., 0., 0.])
assemble shape: (4, 4) 最大误差: 0.0 输出: tensor([[0., 0., 0., 0.],
        [0., 2., 2., 0.],
        [0., 2., 2., 0.],
        [0., 0., 0., 0.]])
transpose shape: (3, 2) 最大误差: 0.0 输出: tensor([[ 1.0028, -0.1669],
        [-0.9893,  0.7299],
        [ 0.5809,  0.4942]])
loop assemble shape: (32, 32, 1, 256) 最大误差: 0.0 样例: tensor([0.6359, 1.0198, 0.5748, 0.3616])


Transform 关注“数据如何被观察、重排、组合和写回”。相关操作可以抽象成几类常见模式：

| 模式 | 代表操作 | 直观理解 |
| --- | --- | --- |
| 连续窗口读取 | `view`、`valid_shape` | 从原 Tensor 某个 offset 开始，取一个规则窗口；边界块只写有效部分 |
| 局部结果写回 | `assemble` | 把小 Tensor 写回大 Tensor 的指定位置 |
| 索引读写 | `gather`、`scatter` | 根据 index Tensor 读出或写入，不要求位置连续 |
| 拼接与类型整理 | `concat`、`cast` | 改变数据组织方式或 dtype，为后续计算准备输入 |
| 维度重排 | `transpose` | 交换维度，常用于让矩阵乘法维度对齐 |
| 分块组合流程 | `loop + view + add + assemble` | 真实算子里常见的“取块、计算、写回”闭环 |

闭环验证聚焦本节最核心的路径：规则窗口读取、边界块、偏移写回、转置，以及循环分块写回。索引读写、拼接和 dtype 整理与这些操作共享同一套阅读方法：先看输入 shape，再看输出 shape，最后用 PyTorch 参考结果验证。


**预期输出说明**

运行成功后，会看到 `transform 模块代表性验证通过`。该单元会打印 `view`、`valid_shape`、`assemble`、`transpose` 和 `loop-view-assemble` 的输出 shape、最大误差和关键输出样例。


## 6. transpose：交换维度

`transpose` 用于交换两个维度。例如：

```python
k_t = pypto.transpose(k, 1, 2)
```

表示交换第 1 维和第 2 维。

在 Attention 中经常会看到 `q @ k.T`。如果不对 Key 做转置，矩阵乘法的维度可能无法对齐。因此，转置不仅是形状变化，也是在为后续计算准备正确的数据布局。


### concat、transpose、cast 与 dtype 复盘

`concat` 用于拼接：

```python
out[:] = pypto.concat([a, b], dim=dim)
```

沿 `dim=0` 拼接时上下合并，沿 `dim=1` 拼接时左右合并。被拼接的维度长度相加，其他维度需要保持一致。

`transpose` 用于交换两个维度：

```python
out[:] = pypto.transpose(x, dim0, dim1)
```

例如 `[B, S, D]` 交换第 1 和第 2 维后得到 `[B, D, S]`。

`cast` 用于转换 dtype：

```python
out[:] = pypto.cast(x, pypto.DT_FP16)
```

常见浮点类型可以这样理解：

| 类型 | 位数 | 特点 |
| --- | --- | --- |
| `FP32` | 32 bit | 精度高、范围大、内存占用较多 |
| `FP16` | 16 bit | 更省内存、速度友好，但范围和精度更低 |
| `BF16` | 16 bit | 范围接近 FP32，小数精度更粗，常用于 AI 计算 |

真实算子里经常会组合使用这些 transform 操作：先 `view/gather` 组织输入，再做 `add/matmul/reduction`，最后用 `concat/scatter/assemble` 写回结果。


`cast`、`concat`、`gather` 和 `scatter` 也属于常见的数据组织操作。它们和 `transpose` 一样，都不一定改变数学计算公式本身，而是在改变数据如何被读取、组合或写回。进入复杂算子后，很多性能和正确性问题并不来自加法或矩阵乘法本身，而是来自这些形状和索引操作是否组织正确。


## 7. 示例：切分 Q/K 并转置 K

示例算子的输入 `x` 形状是 `[B, S, 128]`，最后一维前 64 个元素作为 Q，后 64 个元素作为 K。随后将 K 从 `[B, S, 64]` 转为 `[B, 64, S]`。


## 7.1 Q/K 拆分示例规格

| 项目 | 说明 |
| --- | --- |
| 输入 `x` | FP16 Tensor，shape 为 `[B, S, 128]` |
| `q` | `x[:, :, :64]`，shape 为 `[B, S, 64]` |
| `k` | `x[:, :, 64:128]`，shape 为 `[B, S, 64]` |
| `k_out` | `transpose(k, 1, 2)`，shape 为 `[B, 64, S]` |
| 关键操作 | 切片、转置、输出写回 |

这个示例不是为了完成完整 Attention，而是用于理解矩阵计算前常见的数据组织动作。


In [27]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def split_transpose_kernel(
    x: pypto.Tensor([], pypto.DT_FP16),
    q_out: pypto.Tensor([], pypto.DT_FP16),
    k_out: pypto.Tensor([], pypto.DT_FP16)):
    pypto.set_vec_tile_shapes(2, 8, 64)
    q = x[:, :, :64]
    k = x[:, :, 64:128]
    q_out.move(q)
    k_out.move(pypto.transpose(k, 1, 2))


def main_split_transpose():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    x = torch.randn((2, 8, 128), dtype=torch.float16, device=device)
    q_out = torch.empty((2, 8, 64), dtype=torch.float16, device=device)
    k_out = torch.empty((2, 64, 8), dtype=torch.float16, device=device)

    split_transpose_kernel(x, q_out, k_out)
    q_ref = x[:, :, :64]
    k_ref = x[:, :, 64:128].transpose(1, 2)

    q_diff = (q_out - q_ref).abs().max().item()
    k_diff = (k_out - k_ref).abs().max().item()
    torch.testing.assert_close(q_out, q_ref, rtol=1e-3, atol=1e-3)
    torch.testing.assert_close(k_out, k_ref, rtol=1e-3, atol=1e-3)
    print("split_transpose_kernel 验证通过")
    print("device:", device, "run_mode:", RUN_MODE)
    print("x shape:", tuple(x.shape), "q_out shape:", tuple(q_out.shape), "k_out shape:", tuple(k_out.shape))
    print("q 最大误差:", q_diff, "k 最大误差:", k_diff)
    print("q_out[0, 0, :4]:", q_out[0, 0, :4].cpu())


main_split_transpose()


split_transpose_kernel 验证通过
device: npu:0 run_mode: 0
x shape: (2, 8, 128) q_out shape: (2, 8, 64) k_out shape: (2, 64, 8)
q 最大误差: 0.0 k 最大误差: 0.0
q_out[0, 0, :4]: tensor([1.6211, 0.3357, 0.5396, 0.8018], dtype=torch.float16)


### 代码细节解释

- `q = x[:, :, :64]`：取最后一维前 64 个元素。
- `k = x[:, :, 64:128]`：取最后一维后 64 个元素。
- `pypto.transpose(k, 1, 2)`：交换第 1 维和第 2 维，使 K 从 `[B, S, 64]` 变成 `[B, 64, S]`。
- `q_out.move(q)` 和 `k_out.move(...)`：分别写回两个输出 Tensor。

切片和转置主要是在组织数据形状，为后续矩阵乘法做准备。


### 为什么这个例子和 Attention 有关

大模型里的 Attention 经常会先得到一大块数据，然后把它切成 Q、K、V。这里虽然只切了 Q 和 K，但思想是一样的：

1. 先从大 Tensor 里切出需要的部分。
2. 再把某些部分转置，让矩阵乘法维度对齐。
3. 最后把整理好的结果写到输出 Tensor。

形状操作不是额外的装饰，而是在为后续 Attention 这类真实算子准备正确的数据布局。


这段代码体现了三类操作：

1. `x[:, :, :64]` 是切片，得到 Q。
2. `x[:, :, 64:128]` 是切片，得到 K。
3. `pypto.transpose(k, 1, 2)` 交换 K 的序列维和隐藏维。

PyTorch 参考实现用于验证 shape 和数值。


完整单元会构造 shape 为 `[2, 8, 128]` 的输入，把最后一维前 64 个位置作为 Q，后 64 个位置作为 K，然后把 K 从 `[2, 8, 64]` 转成 `[2, 64, 8]`。

PyTorch 参考结果写成 `x[:, :, :64]` 和 `x[:, :, 64:128].transpose(1, 2)`。这和 PyPTO kernel 中的切片、转置一一对应。


**预期输出说明**

运行成功后，会看到 `q_out` 和 `k_out` 的 shape。`q_out` 应为 `[2, 8, 64]`，`k_out` 应为 `[2, 64, 8]`，并打印 Q/K 的最大误差。


## 8. Tiling API 覆盖与 runtime 观察

前面几节已经从概念上解释了 Tiling、形状变换、切片和转置。本节补充 Tiling API 的完整示例：Cube Tile 和 Vec Tile 的基础配置、不同配置的结果一致性验证，以及 runtime 观察方法。需要注意：runtime 对比受机器、负载、编译缓存和输入规模影响，不要只凭一次运行时间就判断某个 tile 配置一定最优。

In [28]:
def check_close(name, actual, expected, rtol=1e-3, atol=1e-3):
    assert_allclose(actual.cpu().numpy(), expected.cpu().numpy(), rtol=rtol, atol=atol)
    print(f"{name} verified")
    print("  output shape:", tuple(actual.shape))
    print("  expected shape:", tuple(expected.shape))
    print("  output:", actual.cpu())

### 8.1 Cube Tile 基础配置

覆盖 `cube_tile::test_set_cube_tile_shapes_basic`。同一个 cube tile kernel 分别验证 2x2 矩阵乘法和 4x6 @ 6x4 矩阵乘法。

In [29]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def compute_with_cube_tile_shapes_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_cube_tile_shapes([32, 32], [64, 64], [64, 64])
    out.move(pypto.matmul(a, b, a.dtype))


def test_set_cube_tile_shapes_basic():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    print("=" * 60)
    print("Test: Basic Usage of set_cube_tile_shapes Function")
    print("=" * 60)

    dtype = torch.float32
    a = torch.tensor([[1, 2], [3, 4]], dtype=dtype, device=device)
    b = torch.tensor([[5, 6], [7, 8]], dtype=dtype, device=device)
    expected = torch.tensor([[19, 22], [43, 50]], dtype=dtype, device=device)
    set_shapes = [[32, 32], [64, 64], [64, 64]]
    print("set_shapes:", set_shapes)

    out = torch.empty((a.shape[0], b.shape[1]), dtype=dtype, device=device)
    compute_with_cube_tile_shapes_kernel(a, b, out)
    print("Output:", out)
    print("Expected:", expected)
    check_close("cube_tile::test_set_cube_tile_shapes_basic 2x2", out, expected)

    a = torch.randn((4, 6), dtype=dtype, device=device)
    b = torch.randn((6, 4), dtype=dtype, device=device)
    expected = torch.matmul(a, b)
    out = torch.empty((a.shape[0], b.shape[1]), dtype=dtype, device=device)
    compute_with_cube_tile_shapes_kernel(a, b, out)
    print("Output:", out)
    print("Expected:", expected)
    check_close("cube_tile::test_set_cube_tile_shapes_basic 4x6", out, expected)
    print("OK Basic usage of set_cube_tile_shapes function completed successfully")


test_set_cube_tile_shapes_basic()


Test: Basic Usage of set_cube_tile_shapes Function
set_shapes: [[32, 32], [64, 64], [64, 64]]
Output: tensor([[19., 22.],
        [43., 50.]], device='npu:0')
Expected: tensor([[19., 22.],
        [43., 50.]], device='npu:0')
cube_tile::test_set_cube_tile_shapes_basic 2x2 verified
  output shape: (2, 2)
  expected shape: (2, 2)
  output: tensor([[19., 22.],
        [43., 50.]])


[W623 10:24:35.940801441 ToKernelNpu.cpp:41] Warning: Device do not support double dtype now, dtype cast replace with float. (function operator())


Output: tensor([[ 2.1350,  4.0431,  0.1528,  1.9161],
        [-1.2842, -0.4378, -0.2367,  1.5635],
        [-1.2730,  2.8847, -0.4604,  2.2536],
        [-1.1205, -1.1406,  0.3907,  0.4557]], device='npu:0')
Expected: tensor([[ 2.1350,  4.0431,  0.1528,  1.9161],
        [-1.2842, -0.4378, -0.2367,  1.5635],
        [-1.2730,  2.8847, -0.4604,  2.2536],
        [-1.1205, -1.1406,  0.3907,  0.4557]], device='npu:0')
cube_tile::test_set_cube_tile_shapes_basic 4x6 verified
  output shape: (4, 4)
  expected shape: (4, 4)
  output: tensor([[ 2.1350,  4.0431,  0.1528,  1.9161],
        [-1.2842, -0.4378, -0.2367,  1.5635],
        [-1.2730,  2.8847, -0.4604,  2.2536],
        [-1.1205, -1.1406,  0.3907,  0.4557]])
OK Basic usage of set_cube_tile_shapes function completed successfully


### 8.2 Cube Tile 不同配置的结果一致性

覆盖 `cube_tile::test_set_different_tile_shapes_result`。同一个 batched matmul 分别使用三组 cube tile shape，结果应保持一致。

In [30]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def compute_with_different_cube_tile_shapes_kernel(
    x: pypto.Tensor((2, 2, 2), pypto.DT_FP32),
    y: pypto.Tensor((2, 2, 2), pypto.DT_FP32),
    out1: pypto.Tensor((2, 2, 2), pypto.DT_FP32),
    out2: pypto.Tensor((2, 2, 2), pypto.DT_FP32),
    out3: pypto.Tensor((2, 2, 2), pypto.DT_FP32)):
    print("b: 2, m_batch: 2, k_batch: 2, n_batch: 2")

    pypto.set_cube_tile_shapes([32, 32], [16, 16], [32, 32])
    print("pypto.get_cube_tile_shapes():", pypto.get_cube_tile_shapes())
    out1[:] = pypto.matmul(x, y, x.dtype)

    pypto.set_cube_tile_shapes([32, 32], [16, 64], [32, 128])
    print("pypto.get_cube_tile_shapes():", pypto.get_cube_tile_shapes())
    out2[:] = pypto.matmul(x, y, x.dtype)

    pypto.set_cube_tile_shapes([64, 64], [128, 128], [128, 128])
    print("pypto.get_cube_tile_shapes():", pypto.get_cube_tile_shapes())
    out3[:] = pypto.matmul(x, y, x.dtype)


def test_set_different_tile_shapes_result():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    print("=" * 60)
    print("Test: Impact of Different Tile Shape Settings on Calculation Results")
    print("=" * 60)

    dtype = torch.float32
    a = torch.tensor([[[1, 2], [3, 4]], [[5, 6], [7, 8]]], dtype=dtype, device=device)
    b = torch.tensor([[[5, 6], [7, 8]], [[1, 2], [3, 4]]], dtype=dtype, device=device)
    expected = torch.tensor([[[19, 22], [43, 50]], [[23, 34], [31, 46]]], dtype=dtype, device=device)
    out1 = torch.empty((2, 2, 2), dtype=dtype, device=device)
    out2 = torch.empty((2, 2, 2), dtype=dtype, device=device)
    out3 = torch.empty((2, 2, 2), dtype=dtype, device=device)

    compute_with_different_cube_tile_shapes_kernel(a, b, out1, out2, out3)
    print("out1 == out2:", torch.equal(out1, out2))
    print("out2 == out3:", torch.equal(out2, out3))
    check_close("cube_tile::test_set_different_tile_shapes_result out1", out1, expected)
    check_close("cube_tile::test_set_different_tile_shapes_result out2", out2, expected)
    check_close("cube_tile::test_set_different_tile_shapes_result out3", out3, expected)
    print("OK Impact of different tile shape settings on results completed successfully")


test_set_different_tile_shapes_result()


Test: Impact of Different Tile Shape Settings on Calculation Results
b: 2, m_batch: 2, k_batch: 2, n_batch: 2
pypto.get_cube_tile_shapes(): ([32, 32], [16, 16, 16], [32, 32], False)
pypto.get_cube_tile_shapes(): ([32, 32], [16, 64, 64], [32, 128], False)
pypto.get_cube_tile_shapes(): ([64, 64], [128, 128, 128], [128, 128], False)
out1 == out2: True
out2 == out3: True
cube_tile::test_set_different_tile_shapes_result out1 verified
  output shape: (2, 2, 2)
  expected shape: (2, 2, 2)
  output: tensor([[[19., 22.],
         [43., 50.]],

        [[23., 34.],
         [31., 46.]]])
cube_tile::test_set_different_tile_shapes_result out2 verified
  output shape: (2, 2, 2)
  expected shape: (2, 2, 2)
  output: tensor([[[19., 22.],
         [43., 50.]],

        [[23., 34.],
         [31., 46.]]])
cube_tile::test_set_different_tile_shapes_result out3 verified
  output shape: (2, 2, 2)
  expected shape: (2, 2, 2)
  output: tensor([[[19., 22.],
         [43., 50.]],

        [[23., 34.],
        

### 8.3 Cube Tile 不同配置的 runtime 观察

覆盖 `cube_tile::test_set_different_tile_shapes_runtime`。这里使用带 `b_trans=True` 的 batched matmul，对比两组 cube tile shape 的运行时间。

In [31]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def compute_with_tile_32_kernel(
    a: pypto.Tensor((4, 64, 512), pypto.DT_FP32),
    b: pypto.Tensor((4, 128, 512), pypto.DT_FP32),
    out: pypto.Tensor((4, 64, 128), pypto.DT_FP32)):
    pypto.set_cube_tile_shapes([32, 32], [32, 32], [32, 32])
    out.move(pypto.matmul(a, b, a.dtype, b_trans=True))


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def compute_with_tile_64_kernel(
    a: pypto.Tensor((4, 64, 512), pypto.DT_FP32),
    b: pypto.Tensor((4, 128, 512), pypto.DT_FP32),
    out: pypto.Tensor((4, 64, 128), pypto.DT_FP32)):
    pypto.set_cube_tile_shapes([64, 64], [128, 128], [128, 128])
    out.move(pypto.matmul(a, b, a.dtype, b_trans=True))


def test_set_different_tile_shapes_runtime():
    if device == "cpu":
        print("当前环境未执行 NPU runtime 验证；NPU 环境中可执行本模块。")
        return

    print("=" * 60)
    print("Test: Impact of Different Tile Shape Settings on Runtime")
    print("=" * 60)

    b_rt, m_rt, k_rt, n_rt = 4, 64, 512, 128
    dtype = torch.float32
    a = torch.randn((b_rt, m_rt, k_rt), dtype=dtype, device=device)
    b = torch.randn((b_rt, n_rt, k_rt), dtype=dtype, device=device)
    out1 = torch.empty((b_rt, m_rt, n_rt), dtype=dtype, device=device)
    out2 = torch.empty((b_rt, m_rt, n_rt), dtype=dtype, device=device)

    TEST_TIME = 1
    start = time.perf_counter()
    for _ in range(TEST_TIME):
        compute_with_tile_32_kernel(a, b, out1)
    runtime_1 = time.perf_counter() - start

    start = time.perf_counter()
    for _ in range(TEST_TIME):
        compute_with_tile_64_kernel(a, b, out2)
    runtime_2 = time.perf_counter() - start

    print("runtime_1(pypto.set_cube_tile_shapes([32, 32], [32, 32], [32, 32])):", runtime_1)
    print("runtime_2(pypto.set_cube_tile_shapes([64, 64], [128, 128], [128, 128])):", runtime_2)
    print("OK Impact of different tile shape settings on runtime completed successfully")


test_set_different_tile_shapes_runtime()


Test: Impact of Different Tile Shape Settings on Runtime
runtime_1(pypto.set_cube_tile_shapes([32, 32], [32, 32], [32, 32])): 2.9904230389511213
runtime_2(pypto.set_cube_tile_shapes([64, 64], [128, 128], [128, 128])): 3.5464580609695986
OK Impact of different tile shape settings on runtime completed successfully


### 8.4 Vec Tile 基础配置

覆盖 `vec_tile::test_set_vec_tile_shapes_basic`。示例验证 3D 输入和 4D 输入，并展示 `get_vec_tile_shapes()`。

In [32]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def compute_with_vec_tile_shapes_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32),
    set_shapes: tuple):
    pypto.set_vec_tile_shapes(*set_shapes)
    out.move(pypto.add(a, b))


def test_set_vec_tile_shapes_basic():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    print("=" * 60)
    print("Test: Basic Usage of set_vec_tile_shapes Function")
    print("=" * 60)

    dtype = torch.float32
    a = torch.tensor([[[1, 2, 3], [1, 2, 3]]], dtype=dtype, device=device)
    b = torch.tensor([[[4, 5, 6], [4, 5, 6]]], dtype=dtype, device=device)
    expected = torch.tensor([[[5, 7, 9], [5, 7, 9]]], dtype=dtype, device=device)
    set_shapes = (1, 2, 8)
    print("Rule1: len(set_vec_tile_shapes) == len(vec.shape):", len(set_shapes) == len(a.shape))
    print("Rule2: valid input dims must in [1, 4]")
    out = torch.empty_like(a)
    compute_with_vec_tile_shapes_kernel(a, b, out, set_shapes)
    print("Output:", out)
    print("Expected:", expected)
    get_shapes = pypto.get_vec_tile_shapes()
    print("pypto.get_vec_tile_shapes():", get_shapes)
    print("set_shapes == get_shapes:", set_shapes == get_shapes)
    check_close("vec_tile::test_set_vec_tile_shapes_basic 3D", out, expected)

    a = torch.tensor([[[[1, 2, 3], [4, 5, 6]]]], dtype=dtype, device=device)
    b = torch.tensor([[[[7, 8, 9], [10, 11, 12]]]], dtype=dtype, device=device)
    expected = torch.tensor([[[[8, 10, 12], [14, 16, 18]]]], dtype=dtype, device=device)
    set_shapes = (1, 1, 4, 8)
    out = torch.empty_like(a)
    compute_with_vec_tile_shapes_kernel(a, b, out, set_shapes)
    print("Output:", out)
    print("Expected:", expected)
    check_close("vec_tile::test_set_vec_tile_shapes_basic 4D", out, expected)
    print("OK Basic usage of set_vec_tile_shapes function completed successfully")


test_set_vec_tile_shapes_basic()


Test: Basic Usage of set_vec_tile_shapes Function
Rule1: len(set_vec_tile_shapes) == len(vec.shape): True
Rule2: valid input dims must in [1, 4]
Output: tensor([[[5., 7., 9.],
         [5., 7., 9.]]], device='npu:0')
Expected: tensor([[[5., 7., 9.],
         [5., 7., 9.]]], device='npu:0')
pypto.get_vec_tile_shapes(): []
set_shapes == get_shapes: False
vec_tile::test_set_vec_tile_shapes_basic 3D verified
  output shape: (1, 2, 3)
  expected shape: (1, 2, 3)
  output: tensor([[[5., 7., 9.],
         [5., 7., 9.]]])
Output: tensor([[[[ 8., 10., 12.],
          [14., 16., 18.]]]], device='npu:0')
Expected: tensor([[[[ 8., 10., 12.],
          [14., 16., 18.]]]], device='npu:0')
vec_tile::test_set_vec_tile_shapes_basic 4D verified
  output shape: (1, 1, 2, 3)
  expected shape: (1, 1, 2, 3)
  output: tensor([[[[ 8., 10., 12.],
          [14., 16., 18.]]]])
OK Basic usage of set_vec_tile_shapes function completed successfully


### 8.5 Vec Tile 不同配置的结果一致性

覆盖 `vec_tile::test_set_vec_different_tile_shapes_result`。同一个逐元素加法使用三组 vec tile shape，结果应保持一致。

In [33]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def compute_with_vec_different_tile_shapes_kernel(
    a: pypto.Tensor(),
    b: pypto.Tensor(),
    out1: pypto.Tensor(),
    out2: pypto.Tensor(),
    out3: pypto.Tensor()):
    pypto.set_vec_tile_shapes(1, 2, 8)
    print("pypto.get_vec_tile_shapes():", pypto.get_vec_tile_shapes())
    out1.move(pypto.add(a, b))

    pypto.set_vec_tile_shapes(2, 6, 32)
    print("pypto.get_vec_tile_shapes():", pypto.get_vec_tile_shapes())
    out2.move(pypto.add(a, b))

    pypto.set_vec_tile_shapes(5, 3, 16)
    print("pypto.get_vec_tile_shapes():", pypto.get_vec_tile_shapes())
    out3.move(pypto.add(a, b))


def test_set_vec_different_tile_shapes_result():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    print("=" * 60)
    print("Test: Impact of Different Tile Shape Settings on Calculation Results")
    print("=" * 60)

    dtype = torch.float32
    a = torch.tensor([[[1, 2, 3], [1, 2, 3]]], dtype=dtype, device=device)
    b = torch.tensor([[[4, 5, 6], [4, 5, 6]]], dtype=dtype, device=device)
    expected = torch.tensor([[[5, 7, 9], [5, 7, 9]]], dtype=dtype, device=device)
    out1 = torch.empty_like(a)
    out2 = torch.empty_like(a)
    out3 = torch.empty_like(a)
    compute_with_vec_different_tile_shapes_kernel(a, b, out1, out2, out3)
    print("out1 == out2:", torch.equal(out1, out2))
    print("out2 == out3:", torch.equal(out2, out3))
    check_close("vec_tile::test_set_vec_different_tile_shapes_result out1", out1, expected)
    check_close("vec_tile::test_set_vec_different_tile_shapes_result out2", out2, expected)
    check_close("vec_tile::test_set_vec_different_tile_shapes_result out3", out3, expected)
    print("OK Impact of different tile shape settings on results completed successfully")


test_set_vec_different_tile_shapes_result()


Test: Impact of Different Tile Shape Settings on Calculation Results
pypto.get_vec_tile_shapes(): [1, 2, 8]
pypto.get_vec_tile_shapes(): [2, 6, 32]
pypto.get_vec_tile_shapes(): [5, 3, 16]
out1 == out2: True
out2 == out3: True
vec_tile::test_set_vec_different_tile_shapes_result out1 verified
  output shape: (1, 2, 3)
  expected shape: (1, 2, 3)
  output: tensor([[[5., 7., 9.],
         [5., 7., 9.]]])
vec_tile::test_set_vec_different_tile_shapes_result out2 verified
  output shape: (1, 2, 3)
  expected shape: (1, 2, 3)
  output: tensor([[[5., 7., 9.],
         [5., 7., 9.]]])
vec_tile::test_set_vec_different_tile_shapes_result out3 verified
  output shape: (1, 2, 3)
  expected shape: (1, 2, 3)
  output: tensor([[[5., 7., 9.],
         [5., 7., 9.]]])
OK Impact of different tile shape settings on results completed successfully


### 8.6 Vec Tile 不同配置的 runtime 观察

覆盖 `vec_tile::test_set_vec_different_tile_shapes_runtime`。同一个 4D 逐元素加法使用两组 vec tile shape，并打印 runtime。

In [35]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def compute_with_vec_tile_kernel(
    a: pypto.Tensor(),
    b: pypto.Tensor(),
    out: pypto.Tensor(),
    set_shapes: tuple):
    pypto.set_vec_tile_shapes(*set_shapes)
    out.move(pypto.add(a, b))


def test_set_vec_different_tile_shapes_runtime():
    if device == "cpu":
        print("当前环境未执行 NPU runtime 验证；NPU 环境中可执行本模块。")
        return

    print("=" * 60)
    print("Test: Impact of Different Tile Shape Settings on Runtime")
    print("=" * 60)

    vec_rt_shape = (4, 32, 64, 256)
    dtype = torch.float32
    a = torch.randn(vec_rt_shape, dtype=dtype, device=device)
    b = torch.randn(vec_rt_shape, dtype=dtype, device=device)
    out1 = torch.empty(vec_rt_shape, dtype=dtype, device=device)
    out2 = torch.empty(vec_rt_shape, dtype=dtype, device=device)

    TEST_TIME = 1
    start = time.perf_counter()
    for _ in range(TEST_TIME):
        compute_with_vec_tile_kernel(a, b, out1, (1, 2, 4, 128))
    runtime_1 = time.perf_counter() - start

    start = time.perf_counter()
    for _ in range(TEST_TIME):
        compute_with_vec_tile_kernel(a, b, out2, (2, 4, 8, 256))
    runtime_2 = time.perf_counter() - start

    print("runtime_1(pypto.set_vec_tile_shapes(1, 2, 4, 128)):", runtime_1)
    print("runtime_2(pypto.set_vec_tile_shapes(2, 4, 8, 256)):", runtime_2)
    print("OK Impact of different tile shape settings on runtime completed successfully")


test_set_vec_different_tile_shapes_runtime()


Test: Impact of Different Tile Shape Settings on Runtime
runtime_1(pypto.set_vec_tile_shapes(1, 2, 4, 128)): 4.456205532012973
runtime_2(pypto.set_vec_tile_shapes(2, 4, 8, 256)): 2.10978671599878
OK Impact of different tile shape settings on runtime completed successfully


### 8.7 Tiling API 小结

本节完整覆盖了 `tiling_config.py` 中的 beginner 示例。Cube Tile 面向矩阵乘法，Vec Tile 面向逐元素/规约等向量类计算；不同 tile shape 不应改变数学结果，但可能影响 runtime。

实际调优时，先用 PyTorch 参考结果保证正确性，再观察 runtime。不要只凭一次运行时间就判断某个 tile 配置一定最优。

## 9. 课后练习

练习目标是把输入 Tensor `x` 的最后一维切成三段，分别作为 `q`、`k`、`v`：

```python
q = x[:, :, :64]
k = x[:, :, 64:128]
v = x[:, :, 128:192]
```

然后将 `k` 转置后输出。

自测题：

1. Tiling 会改变数学结果吗？
2. reshape 前后必须满足什么条件？
3. Attention 中为什么经常需要转置 Key？

参考答案：

1. 不会，Tiling 只影响执行组织。
2. 元素总数必须一致。
3. 为了让 `q @ k.T` 的矩阵乘法维度对齐。


## 10. 可运行课后练习：切分 Q/K/V

可验证实现从最后一维切出 Q、K、V 三段，每段长度 64；其中 K 需要转置为 `[B, D, S]`，Q 和 V 保持 `[B, S, D]`。


In [36]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def split_qkv_kernel(
    x: pypto.Tensor([], pypto.DT_FP16),
    q_out: pypto.Tensor([], pypto.DT_FP16),
    k_out: pypto.Tensor([], pypto.DT_FP16),
    v_out: pypto.Tensor([], pypto.DT_FP16)):
    pypto.set_vec_tile_shapes(2, 8, 64)
    q = x[:, :, :64]
    k = x[:, :, 64:128]
    v = x[:, :, 128:192]
    q_out.move(q)
    k_out.move(pypto.transpose(k, 1, 2))
    v_out.move(v)


def main_split_qkv():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    x = torch.randn((2, 8, 192), dtype=torch.float16, device=device)
    q_out = torch.empty((2, 8, 64), dtype=torch.float16, device=device)
    k_out = torch.empty((2, 64, 8), dtype=torch.float16, device=device)
    v_out = torch.empty((2, 8, 64), dtype=torch.float16, device=device)

    split_qkv_kernel(x, q_out, k_out, v_out)
    q_ref = x[:, :, :64]
    k_ref = x[:, :, 64:128].transpose(1, 2)
    v_ref = x[:, :, 128:192]

    q_diff = (q_out - q_ref).abs().max().item()
    k_diff = (k_out - k_ref).abs().max().item()
    v_diff = (v_out - v_ref).abs().max().item()
    torch.testing.assert_close(q_out, q_ref, rtol=1e-3, atol=1e-3)
    torch.testing.assert_close(k_out, k_ref, rtol=1e-3, atol=1e-3)
    torch.testing.assert_close(v_out, v_ref, rtol=1e-3, atol=1e-3)
    print("split_qkv_kernel 验证通过")
    print("device:", device, "run_mode:", RUN_MODE)
    print("q_out shape:", tuple(q_out.shape), "k_out shape:", tuple(k_out.shape), "v_out shape:", tuple(v_out.shape))
    print("最大误差:", {"q": q_diff, "k": k_diff, "v": v_diff})


main_split_qkv()


split_qkv_kernel 验证通过
device: npu:0 run_mode: 0
q_out shape: (2, 8, 64) k_out shape: (2, 64, 8) v_out shape: (2, 8, 64)
最大误差: {'q': 0.0, 'k': 0.0, 'v': 0.0}


Q/K 拆分可以自然扩展为 Q/K/V 拆分。`q = x[:, :, :64]` 取第 0 到 63 位置，`k = x[:, :, 64:128]` 取第 64 到 127 位置，`v = x[:, :, 128:192]` 取第 128 到 191 位置。K 的输出需要 `transpose(1, 2)`，所以 shape 从 `[2, 8, 64]` 变成 `[2, 64, 8]`。


**预期输出说明**

运行成功后，会看到 `split_qkv_kernel 验证通过`，并打印 Q、K、V 三个输出 shape 和各自最大误差。


核心语句是：

```python
q = x[:, :, :64]
k = x[:, :, 64:128]
v = x[:, :, 128:192]
q_out.move(q)
k_out.move(pypto.transpose(k, 1, 2))
v_out.move(v)
```


## 11. 本节小结

本节把 Tiling、形状操作和 Tiling API 覆盖整合在一起。Tiling 与形状操作建立的是"数据组织"视角：真实算子开发不仅要写出计算公式，还要组织 Tensor 的形状和布局。Tile Shape 不改变数学结果，只影响执行组织；不同 tile 配置可能影响 runtime，但不应改变正确性。下一节将把逐元素、规约和广播组合起来，完成第 1 章综合实践：行 Softmax。